In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import random
from datetime import datetime, timedelta
from faker import Faker


In [3]:
print(random.seed(42))

None


In [7]:
import numpy as np
import pandas as pd
from scipy import stats
import random
from datetime import datetime, timedelta
from faker import Faker

# ── Reproductibilité ─────────────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)
fake = Faker()
Faker.seed(42)

# ── Paramètres ──────────────────────────────────────────────────────────────
NB_USERS = 500
INTERESTS = ["fitness", "musique", "technologie", "cuisine", "voyage", "cinema"]

# Paramètres de concentration de la loi de Dirichlet
# Plus alpha est élevé → plus l'intérêt est fréquent
# fitness (3.0) plus répandu que cinema (0.5)
ALPHA = [3.0, 2.5, 2.0, 1.5, 1.0, 0.5]

# ── Génération des probabilités via Dirichlet ────────────────────────────────
INTEREST_PROBS = stats.dirichlet.rvs(alpha=ALPHA, random_state=42)[0]

ACTIONS = {
    "fitness":     ["watched workout video", "bought protein powder",
                    "liked gym post", "joined gym"],
    "musique":     ["watched music video", "bought headphones",
                    "liked song", "attended concert", "streamed album"],
    "technologie": ["watched AI talk", "read tech blog", "bought laptop",
                    "bought smartwatch", "liked AI article",
                    "contributed to open source"],
    "cuisine":     ["watched cooking show", "bought ingredients",
                    "liked recipe post", "tried new recipe",
                    "bought kitchen gadget"],
    "voyage":      ["watched travel vlog", "booked flight", "bought luggage",
                    "liked travel photo", "checked into hotel"],
    "cinema":      ["watched movie", "watched trailer", "bought streaming sub",
                    "rated film", "bought cinema ticket"],
}

# ── Inférence du type d'action ───────────────────────────────────────────────
def infer_action_type(action):
    """Déduit le type d'action à partir du verbe."""
    if isinstance(action, str):
        if action.startswith(("watched", "read", "streamed")):
            return "view"
        if action.startswith("bought") or action == "booked flight":
            return "buy"
        if action.startswith("liked") or action == "rated film":
            return "like"
    return "other"

# ── Génération des profils ────────────────────────────────
def generate_users(n=NB_USERS):
    profiles = []
    for i in range(n):
        age = int(np.clip(np.random.normal(loc=35, scale=10), 18, 65))
        nb_interests = random.randint(1, 6)
        interests = list(np.random.choice(
            INTERESTS, size=nb_interests, replace=False, p=INTEREST_PROBS
        ))

        # Pour chaque intérêt, l'utilisateur ne fait qu'un sous-ensemble
        # des actions disponibles (pas toutes !)
        user_actions = {}
        for interest in interests:
            available = ACTIONS[interest]
            k = random.randint(1, max(1, len(available) // 2))
            user_actions[interest] = random.sample(available, k=k)

        # Générer l'activity_log
        activity_log = []
        for _ in range(random.randint(5, 15)):
            # 80% du temps : activité dans ses intérêts
            # 20% du temps : activité hors intérêts
            if random.random() < 0.80:
                category = random.choice(interests)
            else:
                other = [c for c in INTERESTS if c not in interests]
                if other:
                    category = random.choice(other)
                else:
                    category = random.choice(interests)
  
            # Utiliser le sous-ensemble d'actions de cet utilisateur
            if category in user_actions:
                activity_log.append(random.choice(user_actions[category]))
            else:
                # Hors intérêts : on prend une action au hasard
                activity_log.append(random.choice(ACTIONS[category]))

        profiles.append({
            "name":         fake.name(),
            "age":          age,
            "interests":    interests,
            "activity_log": activity_log,
        })
    return profiles

# ── Conversion en DataFrames pour l'analyse ─────────────────────────────────
def profiles_to_dataframes(profiles):
    """Convertit les profils en 2 DataFrames : users + logs."""
    user_records = []
    log_records  = []
    base_date = datetime(2024, 1, 1)

    for idx, profile in enumerate(profiles):
        # DataFrame utilisateurs
        user_records.append({
            "user_id":   idx + 1,
            "name":      profile["name"],
            "age":       profile["age"],
            "interests": profile["interests"],
        })

        # DataFrame logs
        for action in profile["activity_log"]:
            action_type = infer_action_type(action)
            # Déduire la catégorie de l'action
            category = None
            for cat, actions in ACTIONS.items():
                if action in actions:
                    category = cat
                    break

            delta = timedelta(days=random.randint(0, 180),
                              hours=random.randint(0, 23))
            ts = base_date + delta
            log_records.append({
                "user_id":     idx + 1,
                "action":      action,
                "action_type": action_type,
                "category":    category,
                "timestamp":   ts,
                "hour":        ts.hour,
            })

    users_df = pd.DataFrame(user_records)
    logs_df  = pd.DataFrame(log_records)
    return users_df, logs_df

# ── Injection de bruit (doublons + NaN) ──────────────────────────────────────
def inject_noise(df, dup_fraction=0.03, nan_fraction=0.02):
    """Injecte des doublons et des NaN pour simuler des données réelles."""
    df_noisy = df.copy()

    # ── Doublons ──
    n_dup = int(len(df_noisy) * dup_fraction)
    dup_rows = df_noisy.sample(n=n_dup, random_state=42)
    df_noisy = pd.concat([df_noisy, dup_rows], ignore_index=True)

    # ── Valeurs manquantes ──
    n_nan = int(len(df_noisy) * nan_fraction)
    nan_indices = df_noisy.sample(n=n_nan, random_state=42).index
    for idx in nan_indices:
        col = random.choice(["action", "category", "hour"])
        df_noisy.loc[idx, col] = np.nan

    return df_noisy
def clean_data(users_df, logs_df):
    logs_df = logs_df.drop_duplicates()
    logs_df = logs_df.dropna(subset=["action", "category"])
    users_df["age_normalized"] = (users_df["age"] - users_df["age"].mean()) / users_df["age"].std()
    return users_df, logs_df
# ── Exécution ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Générer les probabilités d'intérêts via Dirichlet
    print("=== Probabilités d'intérêts (Dirichlet) ===")
    for interest, prob in zip(INTERESTS, INTEREST_PROBS):
        print(f"  {interest}: {prob:.4f}")

    # 2. Générer les profils (format dict comme le sujet)
    profiles = generate_users()

    # 3. Afficher un profil exemple
    print("\n=== Exemple de profil ===")
    print(profiles[0])

    # 4. Convertir en DataFrames pour l'analyse
    users_df, logs_df = profiles_to_dataframes(profiles)
    logs_df = inject_noise(logs_df)

    print("\n=== DataFrame Utilisateurs ===")
    print(users_df.head())
    print(f"\nShape : {users_df.shape}")

    print("\n=== DataFrame Logs ===")
    print(logs_df.head())
    print(f"\nShape : {logs_df.shape}")
    print(f"Valeurs manquantes :\n{logs_df.isnull().sum()}")
    print(f"Doublons : {logs_df.duplicated().sum()}")

=== Probabilités d'intérêts (Dirichlet) ===
  fitness: 0.2874
  musique: 0.1589
  technologie: 0.1115
  cuisine: 0.0752
  voyage: 0.0741
  cinema: 0.2928

=== Exemple de profil ===
{'name': 'Allison Hill', 'age': 39, 'interests': [np.str_('cinema'), np.str_('cuisine'), np.str_('fitness'), np.str_('technologie'), np.str_('musique'), np.str_('voyage')], 'activity_log': ['watched movie', 'booked flight', 'streamed album', 'watched movie', 'watched workout video', 'watched workout video', 'watched movie', 'streamed album']}

=== DataFrame Utilisateurs ===
   user_id             name  age  \
0        1     Allison Hill   39   
1        2      Noah Rhodes   33   
2        3  Angie Henderson   29   
3        4    Daniel Wagner   29   
4        5  Cristian Santos   28   

                                           interests  
0  [cinema, cuisine, fitness, technologie, musiqu...  
1                                           [cinema]  
2                    [fitness, musique, technologie]  
3    